<!--
  Licensed to the Apache Software Foundation (ASF) under one or more
  contributor license agreements.  See the NOTICE file distributed with
  this work for additional information regarding copyright ownership.
  The ASF licenses this file to You under the Apache License, Version 2.0
  (the "License"); you may not use this file except in compliance with
  the License.  You may obtain a copy of the License at

       http://www.apache.org/licenses/LICENSE-2.0

  Unless required by applicable law or agreed to in writing, software
  distributed under the License is distributed on an "AS IS" BASIS,
  WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
  See the License for the specific language governing permissions and
  limitations under the License.
-->

<center>
<img src="https://hudi.apache.org/assets/images/hudi-logo-medium.png" alt="Hudi logo" width="100%" height="320"/>
</center>

# Apache Hudi with `hudi-rs` (native Python, no Spark)

[`hudi-rs`](https://github.com/apache/hudi-rs) is the native Rust implementation of Apache Hudi with a Python binding (the **`hudi`** package on PyPI). It reads Hudi tables directly into [Apache Arrow](https://arrow.apache.org/) without starting a Spark/JVM session.

This notebook is **specific to the Spark 4 image** (`apachehudi/spark4-hudi`) — the only image where the `hudi` package is installed. It has two parts:

1. **Write** a Hudi table (two commits) to MinIO using Spark.
2. **Read** it back with `hudi-rs` — snapshot, partition-filtered, time-travel, incremental, and read-optimized queries.

API reference: <https://pypi.org/project/hudi/>.

In [ ]:
from importlib.metadata import version

print("hudi-rs (hudi) version:", version("hudi"))

## Part 1 — Create a Hudi table with Spark

We write a Copy-on-Write table to the MinIO `warehouse` bucket, then apply an update so the table has **two commits** (needed to demonstrate time-travel and incremental queries). We pin `hoodie.write.table.version=6` so the table is broadly compatible with the `hudi-rs` reader.

In [ ]:
%run utils.py

In [ ]:
spark = get_spark_session("Hudi-rs Example")

# s3a:// is the Hadoop scheme Spark uses; hudi-rs reads the same bucket via s3://
TABLE_URI_S3A = "s3a://warehouse/hudi_rs_trips"
columns = ["ts", "uuid", "rider", "driver", "fare", "city"]

hudi_options = {
    "hoodie.table.name": "hudi_rs_trips",
    "hoodie.datasource.write.recordkey.field": "uuid",
    "hoodie.datasource.write.partitionpath.field": "city",
    "hoodie.datasource.write.precombine.field": "ts",
    "hoodie.datasource.write.hive_style_partitioning": "true",
    # Write table version 6 for broad hudi-rs read compatibility.
    "hoodie.write.table.version": "6",
}

# Commit 1: initial load
trips = [
    ("2025-08-10 08:15:30", "uuid-001", "rider-A", "driver-X", 18.50, "san_francisco"),
    ("2025-08-10 09:22:10", "uuid-002", "rider-B", "driver-Y", 22.75, "san_francisco"),
    ("2025-08-10 10:05:45", "uuid-003", "rider-C", "driver-Z", 14.60, "chicago"),
    ("2025-08-10 11:30:00", "uuid-004", "rider-D", "driver-W", 35.20, "new_york"),
]
spark.createDataFrame(trips, columns).write.format("hudi") \
    .options(**hudi_options).mode("overwrite").save(TABLE_URI_S3A)
print("Commit 1 written to", TABLE_URI_S3A)

In [ ]:
# Commit 2: update the fare for uuid-001 (an upsert appends a second commit)
updates = [
    ("2025-08-10 12:00:00", "uuid-001", "rider-A", "driver-X", 99.99, "san_francisco"),
]
spark.createDataFrame(updates, columns).write.format("hudi") \
    .options(**hudi_options).mode("append").save(TABLE_URI_S3A)
print("Commit 2 written (uuid-001 fare updated to 99.99)")

In [ ]:
# hudi-rs does not need Spark/JVM, so we can release the Spark session before reading.
stop_spark_session()

## Part 2 — Read the table with `hudi-rs`

`hudi-rs` uses [`object_store`](https://docs.rs/object_store), which picks up `AWS_*` environment variables. The container already sets the credentials, but `object_store` expects `AWS_ENDPOINT` (the container provides `AWS_ENDPOINT_URL`), so we set it here explicitly. Alternatively, pass these through the builder, e.g. `.with_option("aws_endpoint", "http://minio:9000")`.

In [ ]:
import os

os.environ["AWS_ENDPOINT"] = os.getenv("AWS_ENDPOINT_URL", "http://minio:9000")
os.environ["AWS_ALLOW_HTTP"] = "true"
os.environ.setdefault("AWS_ACCESS_KEY_ID", "admin")
os.environ.setdefault("AWS_SECRET_ACCESS_KEY", "password")
os.environ.setdefault("AWS_REGION", "us-east-1")

### Snapshot query

A snapshot query reads the latest version of the data. We also grab the commit timestamps from the `_hoodie_commit_time` metadata column to drive the time-travel and incremental queries below.

In [ ]:
from hudi import HudiTableBuilder
import pyarrow as pa

TABLE_URI_S3 = "s3://warehouse/hudi_rs_trips"

hudi_table = HudiTableBuilder.from_base_uri(TABLE_URI_S3).build()

snapshot = pa.Table.from_batches(hudi_table.read_snapshot(filters=[]))

# Commit timestamps, oldest first (used by the time-travel / incremental queries).
commits = sorted(set(snapshot.column("_hoodie_commit_time").to_pylist()))
print("Commits:", commits)

snapshot.select(["uuid", "rider", "driver", "fare", "city"]).to_pandas()

### Snapshot query with a partition filter

Filters drive partition + file pruning.

In [ ]:
batches = hudi_table.read_snapshot(filters=[("city", "=", "san_francisco")])
pa.Table.from_batches(batches).select(["uuid", "rider", "fare", "city"]).to_pandas()

### Time-travel query

Read the data as of a specific commit timestamp. Reading as of the **first** commit shows `uuid-001` with its original fare (18.50), before the update.

In [ ]:
first_commit = commits[0]
batches = hudi_table.read_snapshot_as_of(first_commit, filters=[])
pa.Table.from_batches(batches).select(["uuid", "rider", "fare", "city"]).to_pandas()

### Incremental query

Read only the records that changed after a given timestamp. Reading the records *after* the first commit returns just the `uuid-001` update from the second commit.

`read_incremental_records(t1, t2)` reads the range `(t1, t2]`; `read_incremental_records(t1)` reads everything after `t1`.

In [ ]:
batches = hudi_table.read_incremental_records(first_commit)
pa.Table.from_batches(batches).select(["uuid", "rider", "fare", "city"]).to_pandas()

### Read-optimized query (Merge-on-Read)

For Merge-on-Read tables, set `hoodie.read.use.read_optimized.mode` on the builder to read only the compacted base files (skipping log-file merging). Our table is Copy-on-Write, so this returns the same result as the snapshot query — it is shown here for completeness.

In [ ]:
ro_table = (
    HudiTableBuilder
    .from_base_uri(TABLE_URI_S3)
    .with_option("hoodie.read.use.read_optimized.mode", "true")
    .build()
)
pa.Table.from_batches(ro_table.read_snapshot(filters=[])) \
    .select(["uuid", "rider", "fare", "city"]).to_pandas()

## Notes

- **Table-version compatibility**: a given `hudi-rs` release supports a specific set of Hudi table versions. This notebook writes table version 6 (via `hoodie.write.table.version=6`) for maximum reader compatibility. See the [hudi-rs README](https://github.com/apache/hudi-rs) for the supported matrix.
- **Timestamp formats**: time-travel and incremental queries accept the Hudi timeline format (`yyyyMMddHHmmssSSS`), Unix epoch, or ISO-8601 timestamps.
- The resulting Arrow tables integrate directly with pandas, Polars, DuckDB, Daft, Ray, etc. — no Spark required.